# Uncertainties in Gaussian spectra

The goal of this notebook is to gather my thoughts about uncertainties in gaussian spectra, such as encountered in water waves in the ocean (though this will apply to any phenomenon that follows a stochastic gaussian process and is analyzed with a spectrum DFT based method). The demonstrations are not included, to keep the explanation short and readable - the goal is to give a "physicist's explanation" of what one's intuition and main things to keep in mind should be. For more information, see a course on DFT / signal analysis or just ask any AI - they know all of this very well. Typically, this assumes that one already has a diffuse understanding and knows the main notions, and just needs a straigth easy-to-read self-contained main-points writeup. For more details / explanations, read a textbook or ask AI.

## A bit of theory

### Fundamentals of gaussian stochastic signals analysis with DFT / iDFT

We consider a physical phenomenon that follows a stochastic process, stationary over the sampling time consider, resulting in an observable time series $x(t)$, with $t$ the time. The phenomenon is sampled over discrete time samples $t_n$, $n=0..N-1$, equally spaced in time (spacing $dt$, sampling frequency $f_s = 1 / dt$) for simplicity, so $t_n=n dt$ and the total signal duration is $T = N dt$. The underlying stochastic process is unknown - all what we have access to, is an individual realization of the stochastic process, through the discrete time series sample of the $x(t_n)$. In the following, we write like a physicist, and sometimes we write $x_n$ (resp. $X_k$), sometimes we write $x(t_n)$ (resp. $X(f_k)$), referring to the value at a point in time or by index. Similary, $x$ (resp. $X$) may represent either the vector of values, or the transform (or operator) sending from the frequency to the time domain (and vice versa) depending on the context. We may also speak interchangeably of the variance (quadratic in the quantity), and standard deviation (the square root of the variance, homogeneous in the quantity).

Given the time series sample, one can build an estimator for the underlying stochastic process complex frequency coefficients (the "true" underlying process from which the individual time series realization is extracted is typically hidden and can never be observed) by using the discrete fourier transform (DFT) estimator for frequency $f_k = k / T$ (as a note, this formula is only strictly valid for $k=0..\lfloor N/2 \rfloor$; for $k=(\lfloor (N+1)/2) \rfloor..N-1$, the frequency wrap up so the frequency effectively sampled is the negative frequency $f_k = (k-N) / T$):

$X(f_k) = \frac{1}{\sqrt{N}} \sum_{n=0}^{N-1} x(t_n) e^{-i 2 \pi k n / N}$, with $k=0..N-1$.

The inverse DFT (iDFT) is then:

$x(t_n) = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} X(f_k) e^{i 2 \pi k n / N}$, with $n=0..N-1$,

so that considering these as transformations of an the input vector of samples: $x \circ X = Id$, with $Id$ the identity.

Note that the $x(t_n)$ and $X(f_k)$ are in general complex. In the case when $x$ is real, $X$ is hermitian, i.e. $X(k) = X^*( [N-k] mod N)$ (the $mod N$ covers the special case that $X(0)$ is real, and that there is no $X(N)$). This means one needs to remember to take absolute norms and consider both real and complex parts of both $x$ and $X$ as needed depending on the case considered.

The Parseval theorem guarantees to us that $\sum_{n=0}^{N-1} |x(t_n)|^2 = \sum_{k=0}^{N-1} |X(f_k)|^2$ . This means that, effectively, the DFT and iDFT with proper normalization conserve energy, and the signal $x$ shows the distribution of signal energy across time, while $X$ shows the distribution of signal energy across frequencies.

Note that the conventions on the DFT and inverse DFT (sign, normalization coefficients splitting between the DFT and iDFT, how the $N$ samples in time or frequency are stored as arrays and how the negative frequency are wrapped, specific behavior for real / hermitian inputs, normalization so the energy is either the energy content per bin or the energy density per bin, etc) can change between textbooks / software packages etc: read the documentation of your package and have consistency tests!

Using the DFT above, one can formulate a discrete estimator $\hat{S}$ for the underlying, hidden power spectrum of the stochastic process ($S(f)$), called the periodogram, defined as: $\hat{S_k} = |X(k)|^2$ . Note that $\hat(S)$ is only an estimator (see more about its properties below)!

Also note that there are "balances" / "equivalences" between how things are sampled in time and space: given $T = N dt$ the duration of the time series sampled, $N$ the number of samples points, $dt$ the sampling interval time, $f_s = 1 / dt$ the sampling rate, $df$ the DFT resolution:

- the highest frequency one can get an estimator for depends only on the sample rate (Nyquist frequency): $f_{max} = f_s / 2$
- the frequency resolution for the estimator depends only on the duration of the signal: $df = 1 / T$
- the ratio between the two, which is the number of unique frequency bins, is the number of samples. For a real signal $x$, the DFT $X$ is hermitian, and there are only half as many unique frequency bins.

In practice, the formula for the DFT and iDFT presented above are not how these are computed - these formula would be quadratic (complexity $\mathcal{O}(n^2)$ for computing the full array transform), while the discrete version of the fast fourier transform (FFT) uses divide-and-conquer radix-based approaches to perform the same computation (implemented differently) in log-linear time (complexity $\mathcal{O}(n \log(n))$), with different prefactors depending on signal length and radix implementation - use a dedicated package, typically wrapping FFTW or similar!

### Uncertainties on the DFT spectrum estimator

Often, a goal in signal analysis is to estimate the power spectrum of a stochastic signal (formally defined as the continuous Fourier transform of its continuous autocorrelation function), and the periodogram $\hat{S}$ is used as an estimator (possibly with taping, windowing, and averaing - see the next section) for that. As highlighted above, this is only an estimator, so it comes with uncertainties. In the following, we will consider that the underlying stochastic process is gaussian. This is generally a reasonable first approximation / model, as gaussianity naturally arises in many processes due to the central limit theory (i.e., the average summation of very many independent identically distributed variables converges towards a gaussian distribution), though it may break down if some nonlinearity, coupling, etc, arises between the variables considered (for example, it is well established empirically that the stock market is usually log-normal, but this does not hold when a major event happens and individual stocks start responding in unison to a specific shock).

The periodogram $\hat{S}$ is asymptotically unbiased and converges to the true underlying power spectrum $S$ of the stochastic process in the following sense:

$f_s \ctime S(f_k) = lim_{T \to \infty} \mathbb{E} [\hat{S}]$ .

Note that this convergence requires 2 ingredients that are challenging to reach in practice: i) the timeseries duration has to tend to infinity (which is challenging, as then the process usually does not remain stationary), ii) one has to consider the expected value, i.e. this holds for averaging over many realizations.

This highlights that $\hat{S}$ is only a stochastic estimator for the underlying spectrum $S$, which is asymptotically unbiased (converges in a given sense under the right conditions). Therefore, one needs to estimate the uncertainty associated with this estimator. Given that the underlying stochastic process is gaussian, the periodogram at each frequency bin is computed from the combination of the 2 real-value coefficients (real part and imaginary part) obtained from the DFT:

$\hat{S)(k) = a_k^2 + b_k^2$, where $a_k$ and $b_k$ are the real and imaginary part of the associated DFT output ($a_k = Re(X_k)$, $b_k = Im(X_k)$).

As the underlying stochastic process is gaussian, this means that taken individually, $a_k$ and $b_k$ follow a gaussian distribution. As a result, $\hat(S)$ mathematically follows a Chi-squared distribution with 2 degrees of freedom. The properties of the Chi-squared distribution are such that:

$var(\hat{S}) = S^2$, so as $\hat{S}$ is our estimator for $S$, the estimator for $var(\hat{S})$ is $\hat{S}^2$, or said otherwise the estimator for the standard deviation $var(\hat{S})$ is $\hat{S}^2$ itself! This means, in practice, that the estimator $\hat{S}$ comes with considerable uncertainty as a result of the nature of the underlying gaussian process, and how this estimator works. This explains why the DFT of a timeseries that is a realization of a gaussian stochastic process is typically very "spiky".

### Tapering, windowning and averaging, combining both in the Welch method estimator, and Welch periodogram uncertainty

#### Tapering

There are some additional assumptions that are implicitly baked into the analysis above. While the mathematics will work for any input signal provided as an array to the DFT or iDFT operator, these are typically discrete variants of the continous Fourier transforms. These typically assume (to make the theory work) / work best (in practice) with periodic signals, and running these with a signal that is not periodic ($x_0 \neq x_{N-1}$) typically result in Gibbs-like oscillatory phenomenon / spectral leakage, meaning energy that should be concentrated in a specific bin, diffuses to neighboring bins.

To avoid this phenomenon, and since most real world signals are not truly periodic, one generally uses tapering: instead of computing:

$X = DFT(x)$,

One computes:

$X_{T} = DFT(T(x)) $, where $T$ is a tapering function. There are several tapering functions available, with different tradeoffs between in particular spectral leaking / amplitude accuracy, vs frequency resolution: typically, one cannot achieve at the same time high frequency resolution (getting narrow lobe where the energy is concentrated) and low spectral leaking (getting energy diffused into side lobes). See for example the tradeoffs between rectangular tapering (equivalent to no windowing), Hann, Hamming, Blackman-Harris, and more - depending on the application, one or another may offer a better tradeoff. These tapering functions typically look like "hats", which make the initial signal go smoothly (or not) to $0$ for the first and last index, while keeping the middle of the array unchanged. In practice, these are often available directly through software packages.

#### Windowing and averaging

A common way to reduce variance of an estimate is to perform averaging - the key result is that, averaging a random variable following a given distribution by sampling it independently $p$ times and doing an equally weighted average, reduces the variance by a factor $1/p$ compared to the individual variance of each sample (this works for any initial distribution, thanks to the central limit theorem). Said otherwise, the standard deviation of the independently-sampled, N-average estimator is $1/\sqrt(p)$ that of the 1-sample estimator.

This can be effectively leveraged in cases when the signal is sampled for significantly longer than the autocorrelation time of the signal, by applying windowing and averaging. For example, if the signal is split into $p$ independent non overlapping segments, and the DFT taken on each of these segments, then these DFTs averaged, the variance of each averaged bin estimate will be reduced by a factor $1/p$ (standard deviation $\sigma$ reduced by $1/\sqrt(p)$): $\sigma(\hat{S}_{p-averaged}) = \sigma(\hat{S}) / \sqrt(p) = \hat{S} / \sqrt(p)$.

Note that increasing the length of the sample signal (remember the sample cannot be so long that the process is no longer stationary though!) and taking the DFT over the whole segment does not reduce the variance of the periodogram - it only increases the frequency resolution. To decrease the variance of the periodogram, one need to perform averaging over segments, meaning that if this is made possible by gathering more consecutive samples, one keeps the same frequency resolution but reduces the variance. So, there is a tradeoff in the windowing and averaging process, between frequency resolution and individual bin estimate variance.

This formula is only valid if consecutive non overlapping segments are independent. This is not the case if, for example, the process has memory over a time scale $\tau$ greater than the segment duration, which can be visible through the autocorrelation function not being uniformly close to $0$ for all time-lags greater than $\tau$. In such a case, the segments are not independent, and the variance reduces slower.

#### The Welch estimator

degrees of freedom

formula for uncertainty

### Handling sensor noise

in real world, sensor noise in addition

for exmaple for mems sensors, looking at acceleration spectrum, typically horizontal support line

2 step: remove sensor noise; compute uncertainties;

### Wrapping it up: how to do proper analysis of a stochastic gaussian process time-series

- plot distribution of samples: gaussian?
- look at autocorrelation function: minimum time scale for which there is no autocorrelation above? segment length, if this is not too little (can also take larger segment length)
- split signal in segments, for each check the mean and std - check that generally stationary
- choose tapering window depending on goal
- apply Welch
- determine noise background
- compute the uncertainty level

## An example: estimates applied for some waves in ice signals

### The data considered

### DFT estimator

### Welch estimator